# TensorMesh Basics — Elements, Basis Functions & Meshes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/basics.ipynb)

Before any PDE: what the building blocks actually look like. This notebook
draws the interpolation nodes and Lagrange basis functions that TensorMesh
generates for each element type and polynomial order, then builds a few
meshes and inspects the `Mesh` object itself.

Nothing here solves anything — it runs in seconds and is the fastest way to
get a feel for the data structures the rest of the gallery is built on.

Docs: [Basics](https://docs.tensor-mesh.com/example_gallery/basics.html) · Source: [`examples/basics/`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/basics/)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Interpolation nodes

`Element.get_basis(order)` returns the node coordinates on the *reference*
element. Raising the order adds mid-edge, mid-face, and interior nodes —
this is what "P2" or "P3" means concretely.

In [ ]:
import matplotlib.pyplot as plt

from tensormesh.element import Line, Triangle, Quadrilateral, Tetrahedron

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for row, element in enumerate([Triangle, Quadrilateral]):
    for col, order in enumerate(range(1, 5)):
        ax = axes[row, col]
        basis = element.get_basis(order)          # [n_basis, dim] reference coords
        for edge in element.points[element.edge]:  # reference-element outline
            ax.plot(edge[:, 0], edge[:, 1], "k-", alpha=0.4)
        ax.scatter(basis[:, 0], basis[:, 1], s=90, zorder=3)
        for i in range(basis.shape[0]):
            ax.text(basis[i, 0], basis[i, 1], f" {i + 1}", fontsize=9)
        ax.set_title(f"{element.__name__} order {order} ({basis.shape[0]} nodes)", fontsize=10)
        ax.set_aspect("equal")
        ax.grid(alpha=0.3)
fig.suptitle("Interpolation nodes on the reference element", fontsize=14)
fig.tight_layout()
plt.show()

## Lagrange basis functions

`Element.get_basis_fns(order)` returns the matching shape functions. Each one
equals 1 at its own node and 0 at every other node — the property that makes
nodal values *be* the solution coefficients.

Shown for linear and quadratic elements; TensorMesh assembles with higher
orders too (the Stokes and cavity notebooks use P2 velocity spaces).

In [ ]:
from tensormesh.element.plot import plot_1d, plot_2d

# 1D: every Lagrange basis function is 1 at its own node and 0 at the others.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for i, order in enumerate([1, 2]):
    plot_1d(Line.get_basis(order), Line.get_basis_fns(order), ax=axes[i], legend=False)
    axes[i].set_title(f"Line, order {order}")
fig.suptitle("1D Lagrange basis functions", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# 2D: the same property, drawn as surfaces over the reference triangle.
fig = plt.figure(figsize=(10, 3.8))
for i, order in enumerate([1, 2]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    plot_2d(Triangle, Triangle.get_basis(order), Triangle.get_basis_fns(order),
            ax=ax, legend=False)
    ax.set_title(f"Triangle, order {order}")
fig.suptitle("2D Lagrange basis functions", fontsize=13)
plt.show()

## Meshes

The `gen_*` constructors call gmsh and return a ready-to-use `Mesh`, with
boundary nodes already flagged. Triangles and quads (and in 3D tets, hexes,
prisms, pyramids) all work with the same assemblers.

In [ ]:
from tensormesh import Mesh

with quiet():
    meshes = {
        "gen_rectangle (tri)": Mesh.gen_rectangle(chara_length=0.12, element_type="tri"),
    "gen_rectangle (quad)": Mesh.gen_rectangle(chara_length=0.12, element_type="quad"),
    "gen_circle": Mesh.gen_circle(chara_length=0.12),
    "gen_L": Mesh.gen_L(chara_length=0.12, element_type="tri"),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, mesh) in zip(axes, meshes.items()):
    mesh.plot(ax=ax)
    ax.set_title(f"{name}\n{mesh.n_points} nodes, {mesh.n_elements} elements", fontsize=10)
    ax.set_aspect("equal")
fig.tight_layout()
plt.show()

## What a `Mesh` holds

`Mesh` extends `torch.nn.Module`: coordinates and connectivity are buffers,
so `.double()` / `.to("cuda")` move an entire problem in one call.

In [ ]:
mesh = meshes["gen_rectangle (tri)"]

print("points          :", tuple(mesh.points.shape), mesh.points.dtype)
print("cell blocks     :", {k: tuple(v.shape) for k, v in mesh.cells.items()})
print("boundary nodes  :", int(mesh.boundary_mask.sum()), "of", mesh.n_points)
print("spatial dim     :", mesh.dim)

# A Mesh is an nn.Module: .double(), .to("cuda"), buffers, the usual PyTorch API.
print("\nafter .double() :", mesh.double().points.dtype)

## Where to next

- [Poisson](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson.ipynb) — put these pieces together into a full solve.
- [The gallery](https://docs.tensor-mesh.com/example_gallery/index.html) — fluid, solid, wave, inverse design.